# GPU-CorruptNet — train on a free Colab T4

1. **Runtime → Change runtime type → GPU (T4)**.
2. Run all cells. The classifier trains on STL-10 substrate + on-the-fly Glitchify-2 corruptions and prints macro-F1 on seen- vs unseen-content test sets.

This is where the *resume number* comes from — an NVIDIA GPU also makes the later latency benchmark (CUDA-event timing, FP16) authentic.

In [ ]:
!git clone https://github.com/tanaymihani/gpu-corruptnet.git
%cd gpu-corruptnet
!pip install -q -e ".[cv,train]"

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime to T4')

In [ ]:
# ResNet-50, full-resolution, more epochs. ~fast on a T4.
!python scripts/train_classifier.py --arch resnet50 --epochs 10 --img-size 224 --batch-size 128 --num-workers 2

In [ ]:
# EfficientNet-B4 for the head-to-head comparison in your resume bullet.
!python scripts/train_classifier.py --arch efficientnet_b4 --epochs 10 --img-size 224 --batch-size 64 --num-workers 2

In [ ]:
import glob, json
for f in sorted(glob.glob('runs/metrics_*.json')):
    m = json.load(open(f))
    print(f"\n{f}")
    for split in ('seen_test', 'unseen_test'):
        s = m[split]
        print(f"  {split:12s} macroF1={s['macro_f1']:.3f} binaryF1={s['binary_f1']:.3f} binaryRecall={s['binary_recall']:.3f}")